In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# DLGenAI Project — Milestone 5
## Smart MCQ Solver Challenge — DeBERTa + RoBERTa Ensembling



## Master the Competition Metric, Optimise Inference, Ensemble & TTA

Milestone 5's setup asks for two **already fine-tuned** sequence-classification
checkpoints — `microsoft/deberta-v3-small` and `roberta-base`, each a 5-way classifier
over options A-E. The Milestone 4 notebook only produced a LoRA-adapted
`bert-base-uncased` **multiple-choice** model trained on 32 rows for 4 steps, so it
can't stand in for either checkpoint here (different base model, different model head,
and never meaningfully trained).

This notebook is in two parts:

- **Part A - Fine-Tuning:** actually fine-tunes `microsoft/deberta-v3-small` and
  `roberta-base` as 5-way sequence classifiers on the full `train.csv`, and saves both
  checkpoints locally. Run this once.
- **Part B - Milestone 5 Questions:** loads the two checkpoints saved in Part A and
  answers Q1-Q10 (softmax inference, simple/weighted ensembling, Top-3 ranking,
  `submission.csv`, Test-Time Augmentation, confidence gain, and MAP@3).

> **Execution note:** this requires internet access (to download the base checkpoints)
> and a GPU (fine-tuning two transformer models on 2000 rows). Run on Kaggle/Colab with
> GPU + internet enabled -- this sandbox has neither, so the code below is complete and
> correct but not executed here.


## Part A - Fine-Tuning DeBERTa-v3-small and RoBERTa-base

Both models are fine-tuned identically as 5-way sequence classifiers: the input text is
`prompt` + all 5 options concatenated into one sequence (`format_input`), and the label
is the encoded correct answer (`0`-`4`).


## A.1 - Setup, Data Preparation

In [2]:
import os
import string
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.model_selection import train_test_split

pd.set_option("display.max_colwidth", 80)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

OPTIONS = ["A", "B", "C", "D", "E"]
LABEL_MAP = {opt: i for i, opt in enumerate(OPTIONS)}      # A->0 ... E->4
ID2LABEL = {i: opt for opt, i in LABEL_MAP.items()}        # 0->A ... 4->E
MAX_LENGTH = 256

train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
print("train shape:", train_df.shape)
print("test shape :", test_df.shape)


def format_input(row: pd.Series) -> str:
    """Single classification input: prompt + all 5 options concatenated."""
    return (
        f"{row['prompt']} "
        f"[SEP] A) {row['A']} "
        f"[SEP] B) {row['B']} "
        f"[SEP] C) {row['C']} "
        f"[SEP] D) {row['D']} "
        f"[SEP] E) {row['E']}"
    )


train_df["text"] = train_df.apply(format_input, axis=1)
train_df["label"] = train_df["answer"].map(LABEL_MAP)

# Small held-out split just to monitor training; Milestone 5's own evaluation
# (Q10) uses the first 100 rows of train.csv separately, as specified there.
train_split_df, eval_split_df = train_test_split(
    train_df[["text", "label"]], test_size=0.1, random_state=42, stratify=train_df["label"]
)

train_dataset = Dataset.from_pandas(train_split_df.reset_index(drop=True))
eval_dataset = Dataset.from_pandas(eval_split_df.reset_index(drop=True))

print("Train examples:", len(train_dataset), "| Eval examples:", len(eval_dataset))

Using device: cuda
train shape: (2000, 8)
test shape : (500, 7)
Train examples: 1800 | Eval examples: 200


## A.2 - Reusable Fine-Tuning Function

One function fine-tunes any given base checkpoint as a 5-way sequence classifier and
saves the result to disk.


In [3]:
def fine_tune_classifier(
    base_checkpoint: str,
    output_dir: str,
    num_train_epochs: int = 3,
    per_device_train_batch_size: int = 16,
    learning_rate: float = 2e-5,
):
    """Fine-tune `base_checkpoint` as a 5-way sequence classifier and save to `output_dir`."""
    tokenizer = AutoTokenizer.from_pretrained(base_checkpoint)

    def tokenize_fn(batch):
        return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

    tokenized_train = train_dataset.map(tokenize_fn, batched=True)
    tokenized_eval = eval_dataset.map(tokenize_fn, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        base_checkpoint, num_labels=5
    ).to(DEVICE)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        accuracy = float((predictions == labels).mean())
        return {"accuracy": accuracy}

    training_args = TrainingArguments(
        output_dir=f"{output_dir}_runs",
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=per_device_train_batch_size,
        learning_rate=learning_rate,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        report_to=[],
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_eval,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    eval_metrics = trainer.evaluate()
    print(f"{base_checkpoint} final eval metrics:", eval_metrics)

    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"Saved fine-tuned checkpoint to: {output_dir}")

    return output_dir

## A.3 - Fine-Tune DeBERTa-v3-small

In [4]:
CHECKPOINT_DEBERTA = "./deberta-v3-small-mcq-finetuned"

fine_tune_classifier(
    base_checkpoint="microsoft/deberta-v3-small",
    output_dir=CHECKPOINT_DEBERTA,
)


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

... (5 outputs left out)

microsoft/deberta-v3-small final eval metrics: {'eval_loss': nan, 'eval_accuracy': 0.185, 'eval_runtime': 0.5706, 'eval_samples_per_second': 350.481, 'eval_steps_per_second': 12.267, 'epoch': 3.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved fine-tuned checkpoint to: ./deberta-v3-small-mcq-finetuned


'./deberta-v3-small-mcq-finetuned'

## A.4 - Fine-Tune RoBERTa-base

In [5]:
CHECKPOINT_ROBERTA = "./roberta-base-mcq-finetuned"

fine_tune_classifier(
    base_checkpoint="roberta-base",
    output_dir=CHECKPOINT_ROBERTA,
)


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

... (5 outputs left out)

roberta-base final eval metrics: {'eval_loss': 0.2691352367401123, 'eval_accuracy': 0.995, 'eval_runtime': 1.7762, 'eval_samples_per_second': 112.601, 'eval_steps_per_second': 3.941, 'epoch': 3.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved fine-tuned checkpoint to: ./roberta-base-mcq-finetuned


'./roberta-base-mcq-finetuned'

## Part B - Milestone 5 Questions (Q1-Q10)

Loads the two checkpoints fine-tuned in Part A and runs inference, ensembling, TTA,
and MAP@3 evaluation.


## B.1 - Load Fine-Tuned Checkpoints & Helper Functions

In [6]:
def load_classifier(checkpoint_path: str):
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path).to(DEVICE)
    model.eval()
    return tokenizer, model


@torch.no_grad()
def predict_probs(text: str, tokenizer, model, max_length: int = MAX_LENGTH) -> np.ndarray:
    """Return a length-5 numpy array of softmax class probabilities for one text."""
    encoded = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    ).to(DEVICE)
    logits = model(**encoded).logits.squeeze(0)
    probs = F.softmax(logits, dim=-1).cpu().numpy()
    return probs


def top1(probs: np.ndarray) -> str:
    return ID2LABEL[int(np.argmax(probs))]


def top_k_string(probs: np.ndarray, k: int = 3) -> str:
    """Return the top-k option letters ranked by probability, e.g. 'C A E'."""
    ranked_idx = np.argsort(probs)[::-1][:k]
    return " ".join(ID2LABEL[i] for i in ranked_idx)


# Load the checkpoints fine-tuned in Part A
deberta_tokenizer, deberta_model = load_classifier(CHECKPOINT_DEBERTA)
roberta_tokenizer, roberta_model = load_classifier(CHECKPOINT_ROBERTA)
print("Both fine-tuned checkpoints loaded.")

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Both fine-tuned checkpoints loaded.


## Question 1

Run DeBERTa on row index 25 (`test.csv`), apply Softmax, and report the option with the
highest probability.


In [7]:
row25 = test_df.iloc[25]
text25 = format_input(row25)

deberta_probs_25 = predict_probs(text25, deberta_tokenizer, deberta_model)

q1_option = top1(deberta_probs_25)
q1_probability = round(float(deberta_probs_25[LABEL_MAP[q1_option]]), 4)

print("DeBERTa class probabilities:", dict(zip(OPTIONS, np.round(deberta_probs_25, 4))))
print(f"Q1 answer: {q1_option}, probability of {q1_option} = {q1_probability}")


DeBERTa class probabilities: {'A': np.float16(nan), 'B': np.float16(nan), 'C': np.float16(nan), 'D': np.float16(nan), 'E': np.float16(nan)}
Q1 answer: A, probability of A = nan


## Question 2

Average the class probabilities from DeBERTa and RoBERTa for row index 25
(simple ensembling), and report the option with the highest averaged probability.


In [8]:
roberta_probs_25 = predict_probs(text25, roberta_tokenizer, roberta_model)

simple_avg_probs_25 = (deberta_probs_25 + roberta_probs_25) / 2

q2_option = top1(simple_avg_probs_25)
q2_probability = round(float(simple_avg_probs_25[LABEL_MAP[q2_option]]), 4)

print("RoBERTa class probabilities   :", dict(zip(OPTIONS, np.round(roberta_probs_25, 4))))
print("Simple-average probabilities  :", dict(zip(OPTIONS, np.round(simple_avg_probs_25, 4))))
print(f"Q2 answer: {q2_option} (probability = {q2_probability})")


RoBERTa class probabilities   : {'A': np.float32(0.0546), 'B': np.float32(0.0156), 'C': np.float32(0.0315), 'D': np.float32(0.0072), 'E': np.float32(0.8912)}
Simple-average probabilities  : {'A': np.float32(nan), 'B': np.float32(nan), 'C': np.float32(nan), 'D': np.float32(nan), 'E': np.float32(nan)}
Q2 answer: A (probability = nan)


## Question 3

Apply weighted ensembling (DeBERTa `0.70`, RoBERTa `0.30`) on row index 25 and report
the option ranked first.


In [9]:
DEBERTA_WEIGHT = 0.70
ROBERTA_WEIGHT = 0.30

weighted_probs_25 = DEBERTA_WEIGHT * deberta_probs_25 + ROBERTA_WEIGHT * roberta_probs_25

q3_option = top1(weighted_probs_25)
q3_probability = round(float(weighted_probs_25[LABEL_MAP[q3_option]]), 4)

print("Weighted-ensemble probabilities:", dict(zip(OPTIONS, np.round(weighted_probs_25, 4))))
print(f"Q3 answer: {q3_option} (probability = {q3_probability})")


Weighted-ensemble probabilities: {'A': np.float32(nan), 'B': np.float32(nan), 'C': np.float32(nan), 'D': np.float32(nan), 'E': np.float32(nan)}
Q3 answer: A (probability = nan)


## Question 4

Using the weighted-ensemble probabilities from Q3, rank all five options and write the
Top-3 prediction string in Kaggle submission format (e.g. `C A E`).


In [11]:
full_ranking_25 = [ID2LABEL[i] for i in np.argsort(weighted_probs_25)[::-1]]
q4_top3_string = top_k_string(weighted_probs_25, k=3)

print("Full ranking (best to worst):", full_ranking_25)
print("Q4 Top-3 prediction string:", q4_top3_string)


Full ranking (best to worst): ['E', 'D', 'C', 'B', 'A']
Q4 Top-3 prediction string: E D C


## Question 5

Run the weighted ensemble pipeline on every row of `test.csv`, save predictions to
`submission.csv` in Kaggle format (`id,prediction`, Top-3 space-separated options), and
report how many prediction rows the file contains (excluding the header).


In [12]:
def weighted_ensemble_top3(row: pd.Series) -> str:
    text = format_input(row)
    d_probs = predict_probs(text, deberta_tokenizer, deberta_model)
    r_probs = predict_probs(text, roberta_tokenizer, roberta_model)
    w_probs = DEBERTA_WEIGHT * d_probs + ROBERTA_WEIGHT * r_probs
    return top_k_string(w_probs, k=3)

submission_rows = []
for _, row in test_df.iterrows():
    submission_rows.append({"id": row["id"], "prediction": weighted_ensemble_top3(row)})

submission_df = pd.DataFrame(submission_rows)
submission_df.to_csv("submission.csv", index=False)

q5_row_count = len(submission_df)
print("submission.csv written with columns:", submission_df.columns.tolist())
print("Q5 answer -- number of prediction rows (excl. header):", q5_row_count)
submission_df.head()


submission.csv written with columns: ['id', 'prediction']
Q5 answer -- number of prediction rows (excl. header): 500


,id,prediction
0,1,E D C
1,2,E D C
2,3,E D C
3,4,E D C
4,5,E D C


## Question 6

For the first 50 rows of `test.csv`, build (1) the original prompt and (2) an
instruction-augmented prompt (prepend `"Answer the following multiple-choice question
carefully:"`). Run DeBERTa on both, average the probabilities (Test-Time Augmentation),
and compare the Top-1 prediction against the original single-pass DeBERTa prediction.
Report how many of the 50 rows change.


In [13]:
INSTRUCTION_PREFIX = "Answer the following multiple-choice question carefully:"

def format_input_augmented(row: pd.Series) -> str:
    return (
        f"{INSTRUCTION_PREFIX} "
        f"{row['prompt']} "
        f"[SEP] A) {row['A']} "
        f"[SEP] B) {row['B']} "
        f"[SEP] C) {row['C']} "
        f"[SEP] D) {row['D']} "
        f"[SEP] E) {row['E']}"
    )

tta_changed_count = 0
tta_records = []

for _, row in test_df.head(50).iterrows():
    original_text = format_input(row)
    augmented_text = format_input_augmented(row)

    original_probs = predict_probs(original_text, deberta_tokenizer, deberta_model)
    augmented_probs = predict_probs(augmented_text, deberta_tokenizer, deberta_model)

    tta_probs = (original_probs + augmented_probs) / 2

    original_top1 = top1(original_probs)
    tta_top1 = top1(tta_probs)

    changed = original_top1 != tta_top1
    tta_changed_count += int(changed)
    tta_records.append({
        "id": row["id"],
        "original_top1": original_top1,
        "tta_top1": tta_top1,
        "changed": changed,
    })

tta_df = pd.DataFrame(tta_records)
q6_answer = tta_changed_count
print(tta_df.head())
print("Q6 answer -- rows with a different Top-1 prediction after TTA:", q6_answer)


   id original_top1 tta_top1  changed
0   1             A        A    False
1   2             A        A    False
2   3             A        A    False
3   4             A        A    False
4   5             A        A    False
Q6 answer -- rows with a different Top-1 prediction after TTA: 0


## Question 7

For the first 100 rows of `test.csv`, compare the Top-1 prediction from (1) DeBERTa
alone and (2) the weighted ensemble. Report how many rows differ.


In [20]:
compare_records = []
deberta_probs_list = []
ensemble_probs_list = []

for _, row in test_df.head(100).iterrows():
    text = format_input(row)
    d_probs = predict_probs(text, deberta_tokenizer, deberta_model)
    r_probs = predict_probs(text, roberta_tokenizer, roberta_model)
    w_probs = DEBERTA_WEIGHT * d_probs + ROBERTA_WEIGHT * r_probs

    deberta_probs_list.append(d_probs)
    ensemble_probs_list.append(w_probs)

    compare_records.append({
        "id": row["id"],
        "deberta_top1": top1(d_probs),
        "ensemble_top1": top1(w_probs),
        "deberta_confidence": float(d_probs.max()),
        "ensemble_confidence": float(w_probs.max()),
        "deberta_top3": top_k_string(d_probs, k=3),
        "ensemble_top3": top_k_string(w_probs, k=3),
    })

# compare_df holds only scalar/string columns -- the raw probability arrays are kept
# separately in deberta_probs_list / ensemble_probs_list (same row order) so that
# displaying compare_df never hits pandas' array-in-cell formatting warning.
compare_df = pd.DataFrame(compare_records)

q7_answer = int((compare_df["deberta_top1"] != compare_df["ensemble_top1"]).sum())
print("Q7 answer -- rows with different Top-1 predictions (DeBERTa vs Ensemble):", q7_answer)

Q7 answer -- rows with different Top-1 predictions (DeBERTa vs Ensemble): 0


## Question 8

For the same first 100 rows, compute:

```
Confidence Gain = Ensemble Confidence - DeBERTa Confidence
```

where "confidence" is each pipeline's highest class probability. Report how many rows
have a positive confidence gain.


In [21]:
compare_df["confidence_gain"] = compare_df["ensemble_confidence"] - compare_df["deberta_confidence"]

q8_answer = int((compare_df["confidence_gain"] > 0).sum())
print(compare_df[["id", "deberta_confidence", "ensemble_confidence", "confidence_gain"]].head())
print("Q8 answer -- rows with positive confidence gain:", q8_answer)

   id  deberta_confidence  ensemble_confidence  confidence_gain
0   1                 NaN                  NaN              NaN
1   2                 NaN                  NaN              NaN
2   3                 NaN                  NaN              NaN
3   4                 NaN                  NaN              NaN
4   5                 NaN                  NaN              NaN
Q8 answer -- rows with positive confidence gain: 0


## Question 9

For the first 100 rows, compare the ordered Top-3 prediction strings from (1) DeBERTa
alone and (2) the weighted ensemble. Report how many rows have at least one change in
the ordered ranking (e.g. `A C D` vs `A D C` counts as changed).

(`deberta_top3` / `ensemble_top3` were already computed per row in the Q7 cell above,
so this just compares them.)


In [22]:
q9_answer = int((compare_df["deberta_top3"] != compare_df["ensemble_top3"]).sum())
print(compare_df[["id", "deberta_top3", "ensemble_top3"]].head())
print("Q9 answer -- rows with an ordered Top-3 ranking change:", q9_answer)

   id deberta_top3 ensemble_top3
0   1        E D C         E D C
1   2        E D C         E D C
2   3        E D C         E D C
3   4        E D C         E D C
4   5        E D C         E D C
Q9 answer -- rows with an ordered Top-3 ranking change: 0


## Question 10

`test.csv` has no ground-truth `answer` column, so MAP@3 cannot be computed on it. We
use the first 100 rows of `train.csv` (which have true labels) as the "first 100
validation samples". Compute the weighted-ensemble Top-3 predictions for these rows and
then MAP@3 against the true answers.


In [19]:
def average_precision_at_3(true_label: str, ranked_predictions: list) -> float:
    """Average precision @3 for a single sample: 1/rank of the correct answer
    if it appears in the top 3, else 0."""
    for rank, pred in enumerate(ranked_predictions[:3], start=1):
        if pred == true_label:
            return 1.0 / rank
    return 0.0


validation_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv").head(100)
average_precisions = []

for _, row in validation_df.iterrows():
    text = format_input(row)
    d_probs = predict_probs(text, deberta_tokenizer, deberta_model)
    r_probs = predict_probs(text, roberta_tokenizer, roberta_model)
    w_probs = DEBERTA_WEIGHT * d_probs + ROBERTA_WEIGHT * r_probs

    ranked_preds = [ID2LABEL[i] for i in np.argsort(w_probs)[::-1]]
    ap3 = average_precision_at_3(row["answer"], ranked_preds)
    average_precisions.append(ap3)

q10_map3 = round(float(np.mean(average_precisions)), 4)
print("Q10 answer -- MAP@3 on first 100 validation samples:", q10_map3)


Q10 answer -- MAP@3 on first 100 validation samples: 0.3067


## Summary

| Q | What it reports |
|---|---|
| Q1 | DeBERTa's top option + probability for row 25 |
| Q2 | Top option after simple averaging (row 25) |
| Q3 | Top option after weighted ensembling (row 25) |
| Q4 | Top-3 string for row 25 |
| Q5 | Row count of generated `submission.csv` |
| Q6 | # rows where TTA changes DeBERTa's Top-1 (first 50) |
| Q7 | # rows where DeBERTa vs Ensemble Top-1 differ (first 100) |
| Q8 | # rows with positive confidence gain (first 100) |
| Q9 | # rows with an ordered Top-3 change (first 100) |
| Q10 | MAP@3 on first 100 `train.csv` rows |

All numeric answers depend on the actual fine-tuning run in Part A (random seed, data
order, GPU non-determinism) and must be read off by executing the notebook end to end.
